In [1]:
import kagglehub
import os
import pandas as pd
from datasets import load_dataset

c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def download_dataset()->list[str]:
    """
    Download the dataset from Kaggle and return the paths to the files.
    """
    dataset_dir = kagglehub.dataset_download("josephleake/huge-collection-of-reddit-votes")
    paths = []
    for dir_path, _, file_names in os.walk(dataset_dir):
        for file_name in file_names:
            paths.append(os.path.join(dir_path, file_name))
    print(f'File path to votes:\n{paths[0]}')
    print(f'File path to submissions:\n{paths[1]}')
    return paths

def get_dataframe()->tuple[pd.DataFrame]:
    """
    Return a tuple of two pandas.Dataframe: votes and submissions.

    Returns:
        tuple[pd.DataFrame]: a tuple of two dataframes.
    """
    paths = download_dataset()
    votes = pd.read_csv(paths[0], sep='\t')
    submissions = pd.read_csv(paths[1], sep='\t')
    return (votes, submissions)

def view_users_votes(votes:pd.DataFrame):
    view = (
        votes
        .groupby(['USERNAME', 'SUBREDDIT', 'VOTE'])
        .size()                         # count upvotes/downvotes in each group
        .unstack(fill_value=0)          # pivot VOTE labels into columns
        .rename(columns={
            'upvote':   'num_upvotes',
            'downvote': 'num_downvotes'
        })
        .reset_index()                  # turn USERNAME & SUBREDDIT back into columns
    )
    return view

In [3]:
votes, submissions = get_dataframe()

votes_subset = votes[votes['SUBREDDIT'] == 'r/Showerthoughts']
submission_subset = submissions[submissions['SUBREDDIT'] == 'Showerthoughts']

File path to votes:
C:\Users\orang\.cache\kagglehub\datasets\josephleake\huge-collection-of-reddit-votes\versions\1\44_million_reddit_votes\44_million_votes.txt
File path to submissions:
C:\Users\orang\.cache\kagglehub\datasets\josephleake\huge-collection-of-reddit-votes\versions\1\submission_info\submission_info.txt


### Neural network - Multiple Layer Perceptron

A standard multiple layer perceptron (MLP) neural network was used as one of the approaches in designing our recommender.  This is a fairly simply approach where the MLP was used to predict user upvotes at the user level, that is, each "upvote" out of a user's voting history is a "1" and an MLP was training to classify a post as a 1 or 0 based on if the user is likely to upvote.  The contents of the post (which is the title field in the submissions data) is pre-processed using the same steps in the assignment, and then vectorised by tf-idf.

Final model's parameters​:
- MLP from scikit-learn MLPClassifier​
- 2 hidden layers of 128 and 64 nodes​
- SMOTE from imblearn used to address the imbalanced nature of the data​
- Default loss function (log-loss, binary entrophy) and solver (adam)​
- Due to sparse data, dimension reduction techniques were tried

In [5]:
#read in downloaded reddit contents data
import pandas as pd

contents = pd.read_csv('reddit_data.csv')
contents['timestamp'] = pd.to_datetime(contents['created_utc'], unit='s')

# print(contents.head())


In [6]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.metrics import ndcg_score

# download stopwords and set up word pre-processing functions from nltk library
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

# Define preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s'!-]", '', text)     # from Q1, changed to retain apostrophe, hyphen and exclamation mark
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [ps.stem(word) for word in tokens]
    return ' '.join(tokens)




[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\orang\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\orang\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\orang\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
# produce top 20 users by votes (up and down votes)
vote_counts_by_user = votes_subset.groupby(['USERNAME']).size().reset_index(name='count')
vote_counts_by_user = vote_counts_by_user.sort_values(by='count', ascending=False)
top_20_users = vote_counts_by_user.head(20)

# print(top_20_users)

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score
import numpy as np

# define lists to store the results
ndcg_results = {}
ndcg_200_results = {}
precision_200_results = {}
recall_200_results = {}

# define precision and recall @k functions, k of 200 is used across all recommenders
def precision_recall_at_k(y_true, y_scores, k=200):

    y_true = np.ravel(y_true)
    y_scores = np.ravel(y_scores)

    top_k_idx = np.argsort(y_scores)[::-1][:k]
    y_pred = np.zeros_like(y_true)
    y_pred[top_k_idx] = 1

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    return precision, recall


# loop through top 20 users, a list is used to make sure same users are evaluated across all models
for username in ['mguardian_north', 'Raven2002', 'Clen23', 'Reeses2150', 'spockspeare', 'apoeticturtle', 'Mash404', 'locks_are_paranoid', \
                  'daygloviking', 'MingeyMackrel', 'thx1138jr', 'Livelogikal', 'baddonkey', 'uncertainusurper', 'lokier01', 'CubyChris', \
                    'pierrekrahn', 'Adventurous_Guy', 'VerbotenPublish', 'stratman42']:

    votes_user = votes.loc[votes['USERNAME'] == username, ['SUBMISSION_ID', 'VOTE']]

    # merge user vote data to the posts data (which has the contents of the posts)
    merged_df = pd.merge(
        contents,
        votes_user,
        left_on='submission_id',    
        right_on='SUBMISSION_ID',
        how='left'
    )


    merged_df['document'] = merged_df['title'].apply(preprocess_text)
    sorted_df = merged_df.sort_values(by='created_utc')

    vectorizer = TfidfVectorizer(max_features=5000, min_df=3)
    X = vectorizer.fit_transform(sorted_df['document'])
    y = (sorted_df['VOTE'] == 'upvote').astype(int)

    # split training and test, since data is sorted by time, this takes first 80% of posts by time as training
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False  
    )


    # this is the original model, without SMOTE, which performed around 5-10% worse with the metrics used

    # clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=10, random_state=42,verbose=True)
    # clf.fit(X_train, y_train)
    # y_pred = clf.predict(X_test)
    # print(classification_report(y_test, y_pred))
    # y_score = clf.predict_proba(X_test)[:, 1] 
    # ndcg = ndcg_score([y_test], [y_score])
    # print(f"nDCG NN: {ndcg:.4f}")



    # error catching as some users do not have upvotes in the test or validation data, which breaks SMOTE
    try:
        # apply SMOTE to deal with the imbalanced nature of the data
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

        # define and train the MLP Classifier, various hyperparameters tested, the following performed best
        clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=50, random_state=42, verbose=False)
        clf.fit(X_train_resampled, y_train_resampled)

        y_pred = clf.predict(X_test)
        # print(classification_report(y_test, y_pred))

        y_score = clf.predict_proba(X_test)[:, 1] 

        # calculate the evaluation metrics for a user
        ndcg = ndcg_score([y_test], [y_score])
        ndcg_results[username] = ndcg
        ndcg_at_200 = ndcg_score([y_test], [y_score], k=200)
        ndcg_200_results[username] = ndcg_at_200
        p, r = precision_recall_at_k([y_test], [y_score], k=200)
        precision_200_results[username] = p
        recall_200_results[username] = r

    except ValueError as e:
        print(f"Skipping {username} due to SMOTE error: {e}")
        continue

Skipping thx1138jr due to SMOTE error: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fit = 2, n_samples = 2
Skipping Adventurous_Guy due to SMOTE error: The target 'y' needs to have more than 1 class. Got 1 class instead


In [ ]:
# save results to csv files 
ndcg_df = pd.DataFrame.from_dict(ndcg_results, orient='index', columns=['nDCG'])
ndcg_df = ndcg_df.reset_index().rename(columns={'index': 'USERNAME'})
ndcg_df.to_csv('ndcg_results.csv', index=False)
ndcg200_df = pd.DataFrame.from_dict(ndcg_200_results, orient='index', columns=['nDCG@200'])
ndcg200_df = ndcg200_df.reset_index().rename(columns={'index': 'USERNAME'})
ndcg200_df.to_csv('ndcg200_results.csv', index=False)
pre200_df = pd.DataFrame.from_dict(precision_200_results, orient='index', columns=['precision@200'])
pre200_df = pre200_df.reset_index().rename(columns={'index': 'USERNAME'})
pre200_df.to_csv('pre200_df_results.csv', index=False)
rec200_df = pd.DataFrame.from_dict(recall_200_results, orient='index', columns=['recall@200'])
rec200_df = rec200_df.reset_index().rename(columns={'index': 'USERNAME'})
rec200_df.to_csv('rec200_df_results.csv', index=False)

In [ ]:
# testing with SVD matric factorisation, model is much quicker (70minutes vs 7minutes), but performance is around 10% worse, so approach was not adopted
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.decomposition import TruncatedSVD

ndcg_results = {}
ndcg_200_results = {}
# Loop through top 20 users
for username in ['mguardian_north', 'Raven2002', 'Clen23', 'Reeses2150', 'spockspeare', 'apoeticturtle', 'Mash404', 'locks_are_paranoid', \
                 'daygloviking', 'MingeyMackrel', 'thx1138jr', 'Livelogikal', 'baddonkey', 'uncertainusurper', 'lokier01', 'CubyChris', \
                    'pierrekrahn', 'Adventurous_Guy', 'VerbotenPublish', 'stratman42']:

    votes_user = votes.loc[votes['USERNAME'] == username, ['SUBMISSION_ID', 'VOTE']]

    # 2. Merge with contents on submission ID
    # Assuming votes has 'SUBMISSION_ID' and contents has 'submission_id'
    merged_df = pd.merge(
        contents,
        votes_user,
        left_on='submission_id',    
        right_on='SUBMISSION_ID',
        how='left'
    )


    merged_df['document'] = merged_df['title'].apply(preprocess_text)
    sorted_df = merged_df.sort_values(by='created_utc')

    vectorizer = TfidfVectorizer(max_features=5000, min_df=3)
    X_old = vectorizer.fit_transform(sorted_df['document'])
    y = (sorted_df['VOTE'] == 'upvote').astype(int)

    # svd matrix factorisation
    svd = TruncatedSVD(n_components=200)
    X = svd.fit_transform(X_old)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False  
    )


    try:

        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

        clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=50, random_state=42, verbose=False)
        clf.fit(X_train_resampled, y_train_resampled)

        y_pred = clf.predict(X_test)

        y_score = clf.predict_proba(X_test)[:, 1] 

        ndcg = ndcg_score([y_test], [y_score])
        ndcg_results[username] = ndcg
        ndcg_at_200 = ndcg_score([y_test], [y_score], k=200)
        ndcg_200_results[username] = ndcg_at_200

    except ValueError as e:
        print(f"Skipping {username} due to SMOTE error: {e}")
        continue

c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\orang\anaconda3\envs\MIT\Lib\si

Skipping thx1138jr due to SMOTE error: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fit = 2, n_samples = 2


c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\orang\anaconda3\envs\MIT\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Skipping Adventurous_Guy due to SMOTE error: The target 'y' needs to have more than 1 class. Got 1 class instead
